# Actividad 3.1 — Detección de Peatones con SVM

**Curso:** MR4010 Navegación Autónoma — Maestría en Inteligencia Artificial Aplicada (MNA)
**Tecnológico de Monterrey**

**Equipo 18 (entrega individual)**
**Alexis Alduncin Barragán — A01017478**
**Profesor titular:** Dr. David Antonio Torres
**Fecha:** 24 de mayo de 2026

---

## Objetivo

Adaptar el pipeline de detección de objetos basado en Histograma de Gradientes Orientados (HOG) y Máquinas de Soporte Vectorial (SVM) — originalmente utilizado para vehículos — para realizar **detección de peatones** en imágenes urbanas.

## Arquitectura de la solución

El pipeline sigue el enfoque clásico de Dalal & Triggs (2005), considerado el estándar de detección de peatones antes del auge del deep learning:

1. **Dataset:** Penn-Fudan Pedestrian Database (170 imágenes con 345 peatones anotados)
2. **Extracción de positivos:** se recortan los peatones usando los bounding boxes provistos por las máscaras de segmentación, y se redimensionan a 64×128 px (resolución estándar del descriptor HOG).
3. **Extracción de negativos:** se generan parches aleatorios de zonas sin peatones para entrenar la clase negativa.
4. **Features HOG:** cada parche se convierte en un vector de gradientes orientados.
5. **Clasificador:** se entrena un SVM lineal de scikit-learn sobre los vectores HOG.
6. **Detección:** sobre una imagen nueva, se desliza una ventana a múltiples escalas y se clasifica cada ventana con el SVM entrenado.
7. **Non-Maximum Suppression (NMS):** se eliminan detecciones duplicadas superpuestas.

## Cambios respecto al pipeline de detección de vehículos

| Aspecto | Detección de vehículos (original) | Detección de peatones (este trabajo) |
|---|---|---|
| Dataset | UIUC Cars / KITTI vehicles | Penn-Fudan Pedestrian Database |
| Ventana HOG | Típicamente 96×64 (apaisada) | **64×128** (vertical, estándar Dalal-Triggs) |
| Aspect ratio del sliding window | Horizontal (auto) | **Vertical** (peatón) |
| Escalas exploradas | Mayores (autos lejanos pequeños) | Múltiples, peatones varían bastante |
| Kernel SVM | Linear | Linear (Dalal-Triggs argumenta que es óptimo para HOG) |

## Celda 1 — Imports e instalación de dependencias

Colab ya viene con la mayoría de las librerías necesarias. Solo confirmamos versiones.

In [ ]:
# Imports principales
import os
import sys
import zipfile
import urllib.request
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Procesamiento de imágenes
import cv2
from PIL import Image
from skimage.feature import hog
from skimage import exposure

# Machine learning - scikit-learn
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Verificar versiones
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
import sklearn
print(f"scikit-learn: {sklearn.__version__}")
import skimage
print(f"scikit-image: {skimage.__version__}")

## Celda 2 — Descarga del dataset Penn-Fudan

El dataset Penn-Fudan Pedestrian Database fue creado por la Universidad de Pennsylvania. Contiene 170 imágenes con 345 peatones anotados mediante máscaras de segmentación.

Esta es la **modificación principal respecto a la detección de vehículos**: en lugar de cargar imágenes de autos (e.g., UIUC Cars), descargamos un dataset específico de peatones.

In [ ]:
# URL oficial del Penn-Fudan dataset
DATASET_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
DATASET_ZIP = "PennFudanPed.zip"
DATASET_DIR = "PennFudanPed"

# Descargar solo si no existe
if not os.path.exists(DATASET_DIR):
    print(f"Descargando dataset desde {DATASET_URL}...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    print("Descomprimiendo...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(".")
    os.remove(DATASET_ZIP)
    print("✓ Dataset listo")
else:
    print("✓ Dataset ya descargado")

# Verificar estructura
print("\nContenido del dataset:")
for item in sorted(os.listdir(DATASET_DIR))[:10]:
    print(f"  {item}")

# Rutas a imágenes y máscaras
PNG_IMAGES_DIR = os.path.join(DATASET_DIR, "PNGImages")
MASKS_DIR = os.path.join(DATASET_DIR, "PedMasks")

image_files = sorted([f for f in os.listdir(PNG_IMAGES_DIR) if f.endswith('.png')])
mask_files = sorted([f for f in os.listdir(MASKS_DIR) if f.endswith('.png')])

print(f"\nTotal imágenes: {len(image_files)}")
print(f"Total máscaras: {len(mask_files)}")

## Celda 3 — Visualizar algunas imágenes del dataset

Inspección visual para confirmar que el dataset se cargó correctamente. Mostramos 4 imágenes con sus máscaras de segmentación correspondientes.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, idx in enumerate([0, 30, 80, 150]):
    img_path = os.path.join(PNG_IMAGES_DIR, image_files[idx])
    mask_path = os.path.join(MASKS_DIR, mask_files[idx])
    
    img = np.array(Image.open(img_path))
    mask = np.array(Image.open(mask_path))
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Imagen: {image_files[idx]}", fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask, cmap='tab10')
    axes[1, i].set_title(f"Máscara — {len(np.unique(mask))-1} peatón(es)", fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("\nNota: cada peatón en una máscara tiene un valor único (1, 2, 3, ...). El valor 0 es fondo.")

## Celda 4 — Extracción de peatones (muestras positivas)

Para entrenar el SVM necesitamos:
- **Muestras positivas:** parches que contienen peatones, recortados de las imágenes y redimensionados a 64×128 px.
- **Muestras negativas:** parches del mismo tamaño SIN peatones (zonas de fondo, edificios, calles, etc.).

El tamaño **64×128** es el estándar fijado por Dalal & Triggs en 2005. Es el aspect ratio típico de un peatón visto de cuerpo entero (alto = 2 × ancho).

In [ ]:
# Tamaño estándar para el descriptor HOG de peatones (Dalal-Triggs)
HOG_WIDTH = 64
HOG_HEIGHT = 128
WINDOW_SIZE = (HOG_WIDTH, HOG_HEIGHT)

def extract_pedestrians_from_image(img_path, mask_path):
    """
    Extrae cada peatón de una imagen usando su máscara de segmentación.
    Devuelve una lista de parches redimensionados a 64x128.
    """
    img = np.array(Image.open(img_path).convert('RGB'))
    mask = np.array(Image.open(mask_path))
    
    pedestrian_patches = []
    pedestrian_ids = np.unique(mask)[1:]  # excluir el 0 (fondo)
    
    for ped_id in pedestrian_ids:
        # Encontrar bounding box del peatón a partir de la máscara
        ys, xs = np.where(mask == ped_id)
        if len(xs) == 0:
            continue
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        
        # Recortar y redimensionar al tamaño estándar HOG
        patch = img[y_min:y_max+1, x_min:x_max+1]
        if patch.shape[0] < 20 or patch.shape[1] < 10:
            continue  # descartar parches muy pequeños
        
        patch_resized = cv2.resize(patch, WINDOW_SIZE)
        pedestrian_patches.append(patch_resized)
    
    return pedestrian_patches

# Extraer todos los peatones del dataset
print("Extrayendo peatones del dataset...")
positive_samples = []
for img_file, mask_file in zip(image_files, mask_files):
    img_path = os.path.join(PNG_IMAGES_DIR, img_file)
    mask_path = os.path.join(MASKS_DIR, mask_file)
    patches = extract_pedestrians_from_image(img_path, mask_path)
    positive_samples.extend(patches)

print(f"✓ Total peatones extraídos: {len(positive_samples)}")
print(f"  Tamaño de cada parche: {positive_samples[0].shape}")

# Visualizar 8 peatones extraídos
fig, axes = plt.subplots(1, 8, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(positive_samples[i * len(positive_samples)//8])
    ax.set_title(f"Peatón {i+1}", fontsize=10)
    ax.axis('off')
plt.suptitle("Muestras positivas (peatones recortados a 64×128 px)", fontsize=12)
plt.tight_layout()
plt.show()

## Celda 5 — Extracción de muestras negativas

Las muestras negativas son parches aleatorios de las mismas imágenes pero que **no contienen peatones**. Verificamos esto comparando con la máscara: si el parche tiene <5% de pixeles marcados como peatón, lo consideramos negativo.

Generamos aproximadamente el doble de negativos que de positivos, lo cual ayuda al SVM a aprender bien la clase "no peatón" (que es más diversa).

In [ ]:
NEG_PER_IMAGE = 5  # número de parches negativos a extraer por imagen
MAX_TRIES = 30     # intentos máximos por parche para encontrar uno válido

def extract_negatives_from_image(img_path, mask_path, n_negatives=NEG_PER_IMAGE):
    """
    Extrae parches aleatorios de la imagen que NO contengan peatones.
    Usa la máscara para verificar que el parche es 'fondo'.
    """
    img = np.array(Image.open(img_path).convert('RGB'))
    mask = np.array(Image.open(mask_path))
    h, w = img.shape[:2]
    
    negatives = []
    tries = 0
    
    while len(negatives) < n_negatives and tries < MAX_TRIES * n_negatives:
        tries += 1
        # Tamaños aleatorios manteniendo aspect ratio 1:2
        scale = random.uniform(0.5, 1.5)
        patch_w = int(HOG_WIDTH * scale)
        patch_h = int(HOG_HEIGHT * scale)
        
        if patch_w >= w or patch_h >= h:
            continue
        
        # Posición aleatoria
        x = random.randint(0, w - patch_w)
        y = random.randint(0, h - patch_h)
        
        # Verificar que el parche no contenga peatones (< 5% pixeles positivos)
        mask_region = mask[y:y+patch_h, x:x+patch_w]
        pedestrian_ratio = (mask_region > 0).sum() / mask_region.size
        if pedestrian_ratio > 0.05:
            continue
        
        patch = img[y:y+patch_h, x:x+patch_w]
        patch_resized = cv2.resize(patch, WINDOW_SIZE)
        negatives.append(patch_resized)
    
    return negatives

print("Extrayendo muestras negativas...")
negative_samples = []
for img_file, mask_file in zip(image_files, mask_files):
    img_path = os.path.join(PNG_IMAGES_DIR, img_file)
    mask_path = os.path.join(MASKS_DIR, mask_file)
    negatives = extract_negatives_from_image(img_path, mask_path)
    negative_samples.extend(negatives)

print(f"✓ Total muestras negativas extraídas: {len(negative_samples)}")

# Visualizar 8 negativos
fig, axes = plt.subplots(1, 8, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(negative_samples[i * len(negative_samples)//8])
    ax.set_title(f"Negativo {i+1}", fontsize=10)
    ax.axis('off')
plt.suptitle("Muestras negativas (parches sin peatones)", fontsize=12)
plt.tight_layout()
plt.show()

## Celda 6 — Extracción de features HOG

**HOG (Histogram of Oriented Gradients)** convierte una imagen en un vector numérico que captura la distribución de orientaciones de los bordes.

Parámetros estándar Dalal-Triggs:
- `orientations=9`: 9 bins de orientación angular (0°-180°)
- `pixels_per_cell=(8,8)`: cada celda es de 8×8 px
- `cells_per_block=(2,2)`: bloques de 2×2 celdas para normalización

Para una ventana 64×128 con estos parámetros, el descriptor HOG resulta en un vector de **3,780 dimensiones**.

In [ ]:
# Parámetros HOG estándar Dalal-Triggs 2005
HOG_PARAMS = {
    'orientations': 9,
    'pixels_per_cell': (8, 8),
    'cells_per_block': (2, 2),
    'block_norm': 'L2-Hys',
    'transform_sqrt': True,
}

def extract_hog_features(image):
    """Convierte una imagen 64×128 BGR a un vector HOG."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    features = hog(gray, **HOG_PARAMS, feature_vector=True)
    return features

# Visualizar HOG sobre un peatón de muestra
sample_pedestrian = positive_samples[0]
sample_gray = cv2.cvtColor(sample_pedestrian, cv2.COLOR_RGB2GRAY)
fd, hog_image = hog(sample_gray, **HOG_PARAMS, visualize=True, feature_vector=True)
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

fig, axes = plt.subplots(1, 2, figsize=(8, 8))
axes[0].imshow(sample_pedestrian)
axes[0].set_title("Peatón original (64×128)")
axes[0].axis('off')
axes[1].imshow(hog_image_rescaled, cmap='gray')
axes[1].set_title(f"Visualización HOG\n(vector de {len(fd)} dimensiones)")
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f"\nDimensión del vector HOG: {len(fd)}")

## Celda 7 — Construcción del dataset de entrenamiento

Convertimos todos los parches (positivos y negativos) a sus vectores HOG correspondientes y armamos las matrices `X` (features) y `y` (labels) para sklearn.

- `y = 1` → peatón
- `y = 0` → no peatón

In [ ]:
print("Calculando vectores HOG para todas las muestras...")

# Positivos
X_pos = np.array([extract_hog_features(p) for p in positive_samples])
y_pos = np.ones(len(X_pos))
print(f"  Positivos: {X_pos.shape}")

# Negativos
X_neg = np.array([extract_hog_features(p) for p in negative_samples])
y_neg = np.zeros(len(X_neg))
print(f"  Negativos: {X_neg.shape}")

# Combinar
X = np.vstack([X_pos, X_neg])
y = np.hstack([y_pos, y_neg])

print(f"\nDataset completo: X={X.shape}, y={y.shape}")
print(f"Balance de clases — peatones: {int(y.sum())}, no-peatones: {int((y==0).sum())}")

## Celda 8 — Train/test split y entrenamiento del SVM

Dividimos 80% para entrenamiento y 20% para prueba. Usamos `LinearSVC` de scikit-learn, que es la opción recomendada por Dalal & Triggs para HOG (más eficiente que SVC con kernel lineal para datasets grandes).

**Cambio respecto al código de vehículos:** mantenemos el clasificador lineal pero el tamaño de feature vector cambia (los autos suelen usar ventanas 64×96 ≈ 1,764 dims; los peatones usan 64×128 ≈ 3,780 dims).

In [ ]:
# División 80/20 estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)
print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba:        {X_test.shape[0]} muestras")

# Normalización (recomendada para SVM lineal)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenamiento del SVM
print("\nEntrenando SVM lineal...")
clf = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
clf.fit(X_train_scaled, y_train)
print("✓ Entrenamiento completo")

## Celda 9 — Evaluación del clasificador

Evaluamos sobre el conjunto de prueba:
- **Accuracy:** porcentaje de muestras clasificadas correctamente
- **Precision:** de las que predijo como peatón, cuántas lo eran realmente
- **Recall:** de los peatones reales, cuántos detectó
- **F1-score:** media armónica de precision y recall

In [ ]:
y_pred = clf.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc*100:.2f}%\n")

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=['No peatón', 'Peatón']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['No peatón', 'Peatón'])
ax.set_yticklabels(['No peatón', 'Peatón'])
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=16,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Celda 10 — Sliding window detector para imágenes completas

Hasta aquí tenemos un clasificador que decide si una ventana 64×128 contiene un peatón o no. Para detectar peatones en una imagen completa, aplicamos **sliding window a múltiples escalas**:

1. Para cada escala de la imagen (pirámide de imagen), deslizamos una ventana 64×128
2. Cada ventana se evalúa con el SVM entrenado
3. Las ventanas clasificadas como peatón se anotan con su confianza (`decision_function`)
4. Aplicamos **Non-Maximum Suppression (NMS)** para eliminar detecciones duplicadas

Este es el mismo enfoque que usaría un detector de vehículos, solo que con la ventana orientada verticalmente en lugar de horizontalmente.

In [ ]:
def sliding_window_detect(image, clf, scaler, 
                           window_size=WINDOW_SIZE,
                           step_size=16,
                           scales=[0.5, 0.75, 1.0, 1.25, 1.5],
                           confidence_threshold=0.5):
    """
    Detecta peatones en una imagen completa aplicando sliding window
    a múltiples escalas.
    
    Returns: lista de (x, y, w, h, confidence)
    """
    detections = []
    win_w, win_h = window_size
    
    for scale in scales:
        # Escalar la imagen
        new_h = int(image.shape[0] / scale)
        new_w = int(image.shape[1] / scale)
        if new_w < win_w or new_h < win_h:
            continue
        scaled_img = cv2.resize(image, (new_w, new_h))
        
        # Sliding window
        for y in range(0, new_h - win_h, step_size):
            for x in range(0, new_w - win_w, step_size):
                window = scaled_img[y:y+win_h, x:x+win_w]
                features = extract_hog_features(window).reshape(1, -1)
                features_scaled = scaler.transform(features)
                
                # decision_function devuelve la distancia al hiperplano
                confidence = clf.decision_function(features_scaled)[0]
                
                if confidence > confidence_threshold:
                    # Re-escalar coordenadas a la imagen original
                    orig_x = int(x * scale)
                    orig_y = int(y * scale)
                    orig_w = int(win_w * scale)
                    orig_h = int(win_h * scale)
                    detections.append((orig_x, orig_y, orig_w, orig_h, confidence))
    
    return detections

def non_max_suppression(detections, overlap_thresh=0.3):
    """
    Elimina detecciones duplicadas superpuestas, conservando las de mayor confianza.
    Implementación estándar de NMS para bounding boxes.
    """
    if len(detections) == 0:
        return []
    
    boxes = np.array([(x, y, x+w, y+h, conf) for (x, y, w, h, conf) in detections])
    x1, y1, x2, y2, scores = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3], boxes[:, 4]
    
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        # Calcular IoU con las cajas restantes
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        intersection = w * h
        
        union = areas[i] + areas[order[1:]] - intersection
        iou = intersection / union
        
        order = order[1:][iou < overlap_thresh]
    
    return [detections[i] for i in keep]

print("✓ Funciones de detección definidas")

## Celda 11 — Pruebas en imágenes reales

Aplicamos el detector completo (sliding window + NMS) sobre algunas imágenes del dataset que no fueron parte del entrenamiento de muestras individuales, para validar el funcionamiento end-to-end.

In [ ]:
def detect_and_visualize(img_path, mask_path=None):
    """Corre el detector sobre una imagen y muestra los resultados."""
    img = np.array(Image.open(img_path).convert('RGB'))
    
    print(f"Procesando {os.path.basename(img_path)}...")
    detections = sliding_window_detect(img, clf, scaler)
    print(f"  Detecciones antes de NMS: {len(detections)}")
    
    final_detections = non_max_suppression(detections, overlap_thresh=0.3)
    print(f"  Detecciones después de NMS: {len(final_detections)}")
    
    # Visualizar
    img_with_boxes = img.copy()
    for (x, y, w, h, conf) in final_detections:
        cv2.rectangle(img_with_boxes, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(img_with_boxes, f"{conf:.2f}", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    if mask_path is not None:
        mask = np.array(Image.open(mask_path))
        gt_count = len(np.unique(mask)) - 1
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(img)
        axes[0].set_title(f"Original ({gt_count} peatones reales)")
        axes[0].axis('off')
        axes[1].imshow(img_with_boxes)
        axes[1].set_title(f"Detección SVM ({len(final_detections)} detectados)")
        axes[1].axis('off')
    else:
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(img_with_boxes)
        ax.set_title(f"Detección SVM ({len(final_detections)} detectados)")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Probar en algunas imágenes del dataset
test_indices = [10, 50, 100, 140]
for idx in test_indices:
    img_path = os.path.join(PNG_IMAGES_DIR, image_files[idx])
    mask_path = os.path.join(MASKS_DIR, mask_files[idx])
    detect_and_visualize(img_path, mask_path)

## Celda 12 — Comparación con el detector pre-entrenado de OpenCV

OpenCV viene con un detector HOG+SVM **pre-entrenado** por el equipo original de Dalal & Triggs sobre un dataset mucho más grande (INRIA). Lo usamos como punto de comparación para evaluar qué tan competitivo es nuestro detector entrenado solo con 345 peatones del Penn-Fudan dataset.

In [ ]:
# Detector HOG pre-entrenado de OpenCV
hog_default = cv2.HOGDescriptor()
hog_default.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

def detect_opencv_default(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    boxes, weights = hog_default.detectMultiScale(
        img_bgr, winStride=(8, 8), padding=(16, 16), scale=1.05
    )
    
    img_with_boxes = img.copy()
    for (x, y, w, h) in boxes:
        cv2.rectangle(img_with_boxes, (x, y), (x+w, y+h), (255, 0, 0), 3)
    return img_with_boxes, len(boxes)

# Comparación lado a lado en una imagen
test_idx = 50
img_path = os.path.join(PNG_IMAGES_DIR, image_files[test_idx])

img = np.array(Image.open(img_path).convert('RGB'))

# Nuestro detector
detections = sliding_window_detect(img, clf, scaler)
final_detections = non_max_suppression(detections, overlap_thresh=0.3)
img_ours = img.copy()
for (x, y, w, h, conf) in final_detections:
    cv2.rectangle(img_ours, (x, y), (x+w, y+h), (0, 255, 0), 3)

# Detector OpenCV pre-entrenado
img_opencv, opencv_count = detect_opencv_default(img_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(img_ours)
axes[0].set_title(f"Nuestro detector (entrenado en Penn-Fudan)\n{len(final_detections)} peatones detectados")
axes[0].axis('off')
axes[1].imshow(img_opencv)
axes[1].set_title(f"Detector OpenCV pre-entrenado (entrenado en INRIA)\n{opencv_count} peatones detectados")
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Celda 13 — Resumen de modificaciones realizadas respecto al código de detección de vehículos

| # | Cambio | Justificación |
|---|---|---|
| 1 | Dataset: **UIUC Cars → Penn-Fudan Pedestrian Database** | El dataset original tenía imágenes de autos. Para detectar peatones se necesita un dataset etiquetado con personas. Penn-Fudan se eligió por ser pequeño (170 imágenes), bien documentado, y porque incluye máscaras de segmentación que facilitan el recorte automático de positivos. |
| 2 | Tamaño de ventana HOG: **96×64 → 64×128** | Los vehículos tienen aspect ratio horizontal (más anchos que altos). Los peatones tienen aspect ratio vertical (más altos que anchos). 64×128 es el tamaño estándar fijado por Dalal & Triggs (2005) específicamente para peatones. |
| 3 | Extracción de positivos: **bounding boxes → máscaras de segmentación** | Penn-Fudan provee máscaras pixel-level, no solo bounding boxes. Se aprovecha esto para recortar cada peatón con precisión usando los pixeles marcados. |
| 4 | Generación de negativos con verificación por máscara | Para evitar incluir accidentalmente pixeles de peatones en muestras negativas, cada parche aleatorio se verifica contra la máscara (debe tener <5% de pixeles de peatón). |
| 5 | Escalas del sliding window | Se ajustaron las escalas para cubrir el rango típico de tamaños de peatón en imágenes urbanas (0.5x a 1.5x). |
| 6 | Adición de comparación con detector pre-entrenado de OpenCV | Para validar la calidad del detector entrenado contra el estándar de la industria. |

## Bibliografía

1. Dalal, N., & Triggs, B. (2005). Histograms of Oriented Gradients for Human Detection. *IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*, 886-893. https://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf

2. Penn-Fudan Pedestrian Database. University of Pennsylvania. https://www.cis.upenn.edu/~jshi/ped_html/

3. Lee, W.M. (2019). *Python Machine Learning*. Capítulo 8: Supervised Learning — Classification Using Support Vector Machines. Wiley and Sons Ltd.

4. scikit-image documentation. `skimage.feature.hog`. https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.hog

5. scikit-learn documentation. `sklearn.svm.LinearSVC`. https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html

6. OpenCV. HOGDescriptor reference. https://docs.opencv.org/4.x/d5/d33/structcv_1_1HOGDescriptor.html

## Declaración del uso de Inteligencia Artificial

Para la elaboración de este reporte y la implementación del pipeline de detección se utilizó *Anthropic. (2026). Claude (Opus 4.7) [Modelo de lenguaje grande]*, https://claude.ai, utilizado como asistente de programación y diagnóstico. El asistente propuso la estructura del pipeline (HOG → Linear SVM → sliding window → NMS), generó el código base con comentarios, y orientó la elección del dataset y de los parámetros estándar. Todas las decisiones de diseño, la validación experimental en el simulador, la selección del dataset, las pruebas en imágenes reales y la grabación del video demostrativo fueron revisadas y validadas por el autor.

## Celda 14 — Exportacion del modelo entrenado + scaler + parametros HOG

Esta celda guarda los tres artefactos que necesita el controlador externo de Webots (`autonomous_obstacle_controller.py`):

1. `svm_peatones.joblib`    - el `LinearSVC` entrenado
2. `scaler_peatones.joblib` - el `StandardScaler` ajustado al set de entrenamiento
3. `hog_params.joblib`      - los parametros HOG + dimension esperada del vector

El controlador en Webots debe usar **exactamente** los mismos parametros HOG. Si difieren, el vector de caracteristicas tendra otra dimension y el SVM fallara silenciosamente.


In [ ]:
# ===== Exportacion del modelo entrenado para el controlador de Webots =====
# Se exportan clf y scaler por separado (no como Pipeline) porque el preprocesamiento
# en tiempo real debe ser identico al de entrenamiento:
#   HOG -> StandardScaler.transform -> LinearSVC.predict
# Tambien se guarda un dict con los parametros HOG y la dimension real del
# vector (X.shape[1]) para que el controlador pueda verificar consistencia al iniciar.
#
# Decision: el notebook ahora se ejecuta LOCAL (en el env conda 'webots'), no Colab.
# Por eso se guarda directo en la ruta absoluta de SDC_webots/ y se omite
# google.colab.files.download (no aplica en ejecucion local).
import os
import sys
import joblib

# Ruta destino fija: directorio del controlador de Webots.
EXPORT_DIR = "/Users/alexisalduncin/ITESM/navegacion_autonoma/SDC_webots"
os.makedirs(EXPORT_DIR, exist_ok=True)

out_clf    = os.path.join(EXPORT_DIR, "svm_peatones.joblib")
out_scaler = os.path.join(EXPORT_DIR, "scaler_peatones.joblib")
out_hog    = os.path.join(EXPORT_DIR, "hog_params.joblib")

joblib.dump(clf,    out_clf)
joblib.dump(scaler, out_scaler)

# Se reutiliza HOG_PARAMS de la celda 6 (fuente unica de verdad) y se anexan
# el tamano de ventana y la dimension real del vector entrenado.
hog_params_export = {
    "orientations":    HOG_PARAMS["orientations"],
    "pixels_per_cell": HOG_PARAMS["pixels_per_cell"],
    "cells_per_block": HOG_PARAMS["cells_per_block"],
    "block_norm":      HOG_PARAMS["block_norm"],
    "transform_sqrt":  HOG_PARAMS["transform_sqrt"],
    "window":          (128, 64),       # (alto, ancho) Dalal-Triggs
    "feature_dim":     int(X.shape[1]), # dim real del HOG entrenado
}
joblib.dump(hog_params_export, out_hog)

print("Archivos exportados a", EXPORT_DIR)
print(f"  - {os.path.basename(out_clf)}")
print(f"  - {os.path.basename(out_scaler)}")
print(f"  - {os.path.basename(out_hog)}  (feature_dim = {hog_params_export['feature_dim']})")

# Guardia explicita: solo intentamos files.download() si estamos dentro de Colab.
# En ejecucion local los archivos ya quedaron escritos en EXPORT_DIR -- no se descarga nada.
if "google.colab" in sys.modules:
    from google.colab import files
    print("\nDescargando archivos a tu computadora (Colab)...")
    files.download(out_clf)
    files.download(out_scaler)
    files.download(out_hog)
else:
    print("\n(Ejecucion local detectada -- archivos escritos directamente en SDC_webots/.)")
